In [ ]:
# =========================
# 0. INSTALL DEPENDENCIES
# =========================
!pip install -q openai-whisper pydub edge-tts
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# =========================
# 1. IMPORTS
# =========================
import os
import re
from pydub import AudioSegment
from difflib import SequenceMatcher
import subprocess
import whisper
import edge_tts
import asyncio

In [ ]:
# =========================
# 2. CONFIG
# =========================

# Update the paths below to process a different input video.
Input_Video = "/content/WHN_Trailer.mp4"

Audio_File = "/content/WHN_audio.wav"
Clean_Audio = "WHN_clean_track.wav"
Stereo_Audio = "WHN_stereo.wav"

Output_Original = "WHN_output_original.mp4"
Output_Clean = "WHN_output_clean.mp4"
Output_Stereo = "WHN_output_stereo.mp4"

Bad_Words = {
    "fuck": "frick",
    "fucking": "freaking",
    "fucker": "sucker",
    "motherfucker": "mothertrucker",
    "shit": "crap",
    "bitch": "wench",
    "asshole": "butthole",
    "goddamn": "goshdarn",
    "ass": "butt",
    "prick": "jerk"
}

In [ ]:
# =========================
# 3. EXTRACT AUDIO
# =========================
def extract_audio():
    os.system(f"ffmpeg -y -i {Input_Video} -ac 1 -ar 16000 {Audio_File}")

In [ ]:
# =========================
# 4. TRANSCRIBE (WHISPER)
# =========================
def transcribe():
    model = whisper.load_model("medium")
    result = model.transcribe(Audio_File, word_timestamps=True)
    return result

In [ ]:
# =========================
# 5. FUZZY MATCH HELPER
# =========================
# def is_similar(a, b, threshold=0.75):
#     return SequenceMatcher(None, a, b).ratio() >= threshold
    # This function looks for anything that is similar to bad words to catch all possible words that need to be censored
    # I ended up taking this out of my pipeline as it hurt the model more than helped it

In [ ]:
# =========================
# 6. FIND BAD WORDS
# =========================
def find_targets(transcription):
    targets = []

    for segment in transcription["segments"]:
        for word in segment["words"]:
            w = word["word"].lower().strip()
            # Strip punctuation that Whisper sometimes attaches
            w_clean = w.strip(".,!?\"'-")

            matched_key = None
            for bad_word in Bad_Words:
                if w_clean == bad_word:
                    matched_key = bad_word
                    break

            if matched_key:
                targets.append({
                    "word": w_clean,
                    "replacement": Bad_Words[matched_key],
                    "start": word["start"],
                    "end": word["end"]
                })

    return targets

In [ ]:
# =========================
# 7. WHISPER PROXY EVAL
# =========================
def extract_all_bad_words(transcription):
    all_bad = []

    for segment in transcription["segments"]:
        for word in segment["words"]:
            w = re.sub(r"[^\w]", "", word["word"].lower())

            if w in Bad_Words:
                all_bad.append({
                    "word": w,
                    "time": word["start"]
                })

    return all_bad


def evaluate_full_video(transcription, targets):
    all_bad = extract_all_bad_words(transcription)

    matched = []
    missed = []
    false_positives = []

    tolerance = 0.5

    # Match real → predicted
    for real in all_bad:
        found = False
        for pred in targets:
            if abs(pred["start"] - real["time"]) < tolerance:
                matched.append(real)
                found = True
                break
        if not found:
            missed.append(real)

    # Match predicted → real
    for pred in targets:
        found = False
        for real in all_bad:
            if abs(pred["start"] - real["time"]) < tolerance:
                found = True
                break
        if not found:
            false_positives.append(pred)

    precision = len(matched) / len(targets) if targets else 0
    recall = len(matched) / len(all_bad) if all_bad else 0
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "total_bad_words": len(all_bad),
        "detected": len(targets),
        "matched": len(matched),
        "missed": missed,
        "false_positives": false_positives
    }

In [ ]:
# =========================
# 8. (TTS)
# =========================
import edge_tts
import asyncio
from pydub import AudioSegment

async def generate_tts(text, output_file_base):
    mp3_file = f"{output_file_base}.mp3"
    wav_file = f"{output_file_base}.wav"

    # I chose to go with a standard TTS model here instead of a more robust voice cloning option since this is a smaller scale project
    communicate = edge_tts.Communicate(text, voice="en-US-GuyNeural")
    await communicate.save(mp3_file)

    # Convert MP3 → WAV
    audio = AudioSegment.from_file(mp3_file, format="mp3")
    audio.export(wav_file, format="wav")

    return wav_file

In [ ]:
# =========================
# 9. GENERATE CLEAN WORDS
# =========================
async def generate_clean_words_async(targets):
    for i, t in enumerate(targets):
        await generate_tts(
            t["replacement"],
            f"clean_{i}"
        )

In [ ]:
# =========================
# 10. STRETCH AUDIO HELPER
# =========================
def stretch_audio(input_wav, target_duration_ms):
    """Use ffmpeg atempo filter to stretch audio to target duration."""
    input_audio = AudioSegment.from_wav(input_wav)
    current_duration = len(input_audio)

    if target_duration_ms <= 0:
        return input_audio  # just return original

    if current_duration == 0:
        return AudioSegment.silent(duration=target_duration_ms)

    speed_factor = current_duration / target_duration_ms

    # Clamp for ffmpeg atempo limits
    speed_factor = max(0.5, min(2.0, speed_factor))

    output_wav = input_wav.replace(".wav", "_stretched.wav")

    subprocess.run([
        "ffmpeg", "-y", "-i", input_wav,
        "-filter:a", f"atempo={speed_factor}",
        output_wav
    ], capture_output=True)

    result = AudioSegment.from_wav(output_wav)

    # Ensure exact duration
    if len(result) < target_duration_ms:
        result += AudioSegment.silent(duration=target_duration_ms - len(result))
    else:
        result = result[:target_duration_ms]

    return result

In [ ]:
# =========================
# 11. VOLUME MATCH HELPER
# =========================
def match_volume(replacement, original_segment):
    target_dBFS = original_segment.dBFS
    if target_dBFS == float('-inf'):
        return replacement
    change_in_dBFS = target_dBFS - replacement.dBFS
    return replacement.apply_gain(change_in_dBFS)

In [ ]:
# =========================
# 12. CREATE CLEAN AUDIO
# =========================
def create_clean_audio(targets):
    audio = AudioSegment.from_wav(AUDIO_FILE)
    clean_audio = audio

    for i, t in enumerate(reversed(targets)):
        actual_i = len(targets) - 1 - i
        start = int(t["start"] * 1000)
        end   = int(t["end"]   * 1000)
        original_duration = end - start

        replacement = stretch_audio(f"clean_{actual_i}.wav", original_duration)
        replacement = replacement.fade_in(10).fade_out(10)
        clean_audio = clean_audio[:start] + replacement + clean_audio[end:]

    clean_audio.export(CLEAN_AUDIO, format="wav")

In [ ]:
# =========================
# 13. CREATE STEREO
# =========================
def create_stereo():
    original = AudioSegment.from_wav(Audio_File)
    clean = AudioSegment.from_wav(Clean_Audio)

    # Ensure same properties
    clean = clean.set_frame_rate(original.frame_rate)
    clean = clean.set_channels(1)
    original = original.set_channels(1)

    # Match exact length
    min_len = min(len(original), len(clean))
    original = original[:min_len]
    clean = clean[:min_len]

    stereo = AudioSegment.from_mono_audiosegments(original, clean)
    stereo.export(Stereo_Audio, format="wav")

In [ ]:
# =========================
# 14. MERGE VIDEO + AUDIO
# =========================
def merge(video, audio, output):
    os.system(f"""
    ffmpeg -y -i {video} -i {audio} \
    -c:v copy -map 0:v:0 -map 1:a:0 {output}
    """)

In [ ]:
# =========================
# 15. RUN PIPELINE
# =========================
async def run():
    print("Step 1: Extract audio")
    extract_audio()

    print("Step 2: Transcribe")
    transcription = transcribe()
    for segment in transcription["segments"]:
      print(f"[{segment['start']:.2f}s -> {segment['end']:.2f}s] {segment['text']}")
      for word in segment["words"]:
        print(f"  word: '{word['word']}' | start: {word['start']:.3f}s | end: {word['end']:.3f}s")

    print("Step 3: Find targets")
    targets = find_targets(transcription)


    results = evaluate_full_video(transcription, targets)

    print("\n FULL VIDEO EVALUATION")
    print(f"Total bad words: {results['total_bad_words']}")
    print(f"Detected:        {results['detected']}")
    print(f"Matched:         {results['matched']}")

    print(f"\nPrecision: {results['precision']:.2f}")
    print(f"Recall:    {results['recall']:.2f}")
    print(f"F1 Score:  {results['f1']:.2f}")

    print("\n MISSED WORDS:")
    for m in results["missed"]:
        print(f"{m['word']} at {m['time']:.2f}s")

    print("\n FALSE POSITIVES:")
    for fp in results["false_positives"]:
        print(f"{fp['word']} at {fp['start']:.2f}s")

    if not targets:
        print("No bad words found.")
        return

    print("Step 4: Generate clean words")
    await generate_clean_words_async(targets)

    print("Step 5: Create clean audio")
    create_clean_audio(targets)

    print("Step 6: Create stereo")
    create_stereo()

    print("Step 7: Export videos")
    merge(Input_Video, Audio_File, Output_Original)
    merge(Input_Video, Clean_Audio, Output_Clean)
    merge(Input_Video, Stereo_Audio, Output_Stereo)

    print("\n DONE!")

In [ ]:
# =========================
# RUN IT
# =========================
await run()

Step 1: Extract audio
Step 2: Transcribe
[1.20s -> 3.84s]  You're always eating. You never stop eating.
  word: ' You're' | start: 1.200s | end: 1.720s
  word: ' always' | start: 1.720s | end: 2.100s
  word: ' eating.' | start: 2.100s | end: 2.520s
  word: ' You' | start: 2.860s | end: 2.980s
  word: ' never' | start: 2.980s | end: 3.360s
  word: ' stop' | start: 3.360s | end: 3.780s
  word: ' eating.' | start: 3.780s | end: 3.840s
[3.92s -> 4.64s]  I stop eating.
  word: ' I' | start: 3.920s | end: 4.080s
  word: ' stop' | start: 4.080s | end: 4.380s
  word: ' eating.' | start: 4.380s | end: 4.640s
[4.80s -> 8.20s]  Yeah, if you're sleeping or if you're shooting a guy.
  word: ' Yeah,' | start: 4.800s | end: 5.020s
  word: ' if' | start: 5.020s | end: 5.320s
  word: ' you're' | start: 5.320s | end: 5.660s
  word: ' sleeping' | start: 5.660s | end: 6.160s
  word: ' or' | start: 6.160s | end: 6.680s
  word: ' if' | start: 6.680s | end: 6.800s
  word: ' you're' | start: 6.800s | end: 7.1